# Register a model in the MLFlow server

We will use the MlflowClient instance to:

- Register a new version for the experiment green-taxi-trip-duration
- Retrieve the latests versions of the model 
- Transition to "Production" and adding annotations to it.

The environment variables are set in the a `.env` file. Create one and add yours.

In [ ]:
import mlflow
from mlflow.tracking.client import MlflowClient
import os 
from dotenv import load_dotenv
from datetime import datetime

In [ ]:
load_dotenv()
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
RUN_ID = os.environ.get("RUN_ID")

Connect to the MLFlow server:

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

Register the model:

In [ ]:
model_uri = f"runs:/{RUN_ID}/model"
mlflow.register_model(model_uri=model_uri, name="lr-ride-duration")

You now also see the model in the MLFlow UI under `Models`.

In [ ]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

We can use the MlflowClient to get more information about the model:

In [ ]:
model_name = "lr-ride-duration"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

We see we only have a version 1 of the model. If we would have registered more artifacts as models under the same name, we would have seen more versions here.
But we want to change the stage from `None` to `Production` and add a description to it. 

First change the stage:

In [ ]:
model_version = 1
new_stage = "Production"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

And now add the description:

In [ ]:
date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

If we look at the model again the stage change to Production and the description is added:

In [ ]:
latest_versions = client.get_latest_versions(name=model_name)
for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}, description: {version.description}")